#Enhancing Stock Price Prediction with Hybrid LSTM-GRU Architectures and Attention Mechanisms on Grouped Time-Series Data.

In [ ]:

import os
import kagglehub
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow.keras
from tensorflow.keras.utils import plot_model

from tensorflow.keras.layers import Input, LSTM, GRU, Dropout, Dense, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

I began by first replicating the research by *A. Lawi et al* using the same timeframe to ensure that my research aligns with theirs. I used the same start and end dates 04/01/2010 - 04/02/2022 in DD/MM/YYYY format. A period of 12 years and split training and testing 80/20 at 9 years 9 months.

##Data extraction Yahooo API

In [ ]:
# Download stock data from Yahoo finance
tickers = ["AMZN", "GOOGL", "BALL", "QCOM"]
start_date = "2010-01-04"
end_date = "2022-02-04"

data = yf.download(tickers, start=start_date, end=end_date, auto_adjust=False)


# This section is to increase the scale of the data based on stock splitting so that it matches the research done by A. Lawi et al (2022)
# Define the split date and ratio
ball_split_date = pd.Timestamp('2017-05-17') # Ball Stock split date
amzn_split_date = pd.Timestamp('2022-06-06') # AMZN Stock split date
googl_split_date = pd.Timestamp('2022-07-15') # GOOGL Stock split date
ball_split_ratio = 2 #2:1 split
amzn_googl_split_ratio = 20 #20:1 split


# # Multiply prices BEFORE the split date by the split ratio
# data.loc[data.index < ball_split_date, (slice(None), 'BALL')] *= ball_split_ratio
# data.loc[data.index < amzn_split_date, (slice(None), 'AMZN')] *= amzn_googl_split_ratio
# data.loc[data.index < googl_split_date, (slice(None), 'GOOGL')] *= amzn_googl_split_ratio


# Display the first few rows of the data
print(data.tail())

In [ ]:
data.columns

In [ ]:
print(len(data))

The values of AMZN and GOOGL were multiplied by 20 to get the unadjusted prices as GOOGL and AMZN stocks were split 20:1 in May 2022, also BLL (later renamed to BALL) had a 2:1 split on the 17 of May 2017. QCOMs last stock split was in 2004 so it was not included.

In [ ]:
# I'm mainly going to be working with the close prices
# Extract the 'Close' prices for each ticker
close_prices = data['Close']

In [ ]:

colors = {
    "BALL": "grey",
    "GOOGL": "orange",
    "QCOM": "gold"
}

# Plot the closing prices
plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(10, 6))
for ticker in tickers:
    plt.plot(close_prices[ticker], label=ticker, color=colors.get(ticker, None))

plt.title('Closing Prices for Selected Stocks (2010-2022)', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Closing Price (USD)', fontsize=12)
plt.legend(title='Ticker', fontsize=10)
plt.grid(True)
plt.tight_layout()
plt.savefig('close_prices_plot.png')
plt.show()

## 4 Neural Architecture Models (based on prior research)

In [ ]:
# Define model parameters
num_companies = 4
timesteps = 40
num_features = 1
lstm_units = 160 # For a total of 160 units per company
gru_units = 160 # For a total of 160 units per company
downsized_units = 160 # For the downsized models
stabilized_units = 160 # For the stabilized downsized model
dropout_rate = 0.5

# Define inputs for each of the 4 companies
input_amzn = Input(shape=(timesteps, num_features), name='amzn_input')
input_googl = Input(shape=(timesteps, num_features), name='googl_input')
input_bll = Input(shape=(timesteps, num_features), name='bll_input')
input_qcom = Input(shape=(timesteps, num_features), name='qcom_input')

In [ ]:
input_amzn.shape

###LSTM Models

In [ ]:
# Create independent LSTM blocks for each company
def create_company_model(input_tensor):
    x = LSTM(lstm_units, return_sequences=True)(input_tensor)
    x = Dropout(dropout_rate)(x)
    x = LSTM(lstm_units, return_sequences=True)(x)
    x = Dropout(dropout_rate)(x)
    return x

####Model 1

In [ ]:
amzn_branch = create_company_model(input_amzn)
googl_branch = create_company_model(input_googl)
bll_branch = create_company_model(input_bll)
qcom_branch = create_company_model(input_qcom)

# Concatenate the outputs from all company branches
# The shape of the concatenated output will be (None, 40, 640)
# (4 branches * 160 units/branch = 640 total features)
concatenated_output = Concatenate()([amzn_branch, googl_branch, bll_branch, qcom_branch])

# Create separate outputs for each company as requested
amzn_final = LSTM(lstm_units, name='amzn_0')(concatenated_output)
amzn_final = Dense(1, name='amzn_output')(amzn_final)

googl_final = LSTM(lstm_units, name='googl_0')(concatenated_output)
googl_final = Dense(1, name='googl_output')(googl_final)

bll_final = LSTM(lstm_units, name='bll_0')(concatenated_output)
bll_final = Dense(1, name='bll_output')(bll_final)

qcom_final = LSTM(lstm_units, name='qcom_0')(concatenated_output)
qcom_final = Dense(1, name='qcom_output')(qcom_final)

# Define the full model with all inputs and outputs
Lmodel1 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

adam = Adam(learning_rate=0.001)

# Compile the model
Lmodel1.compile(optimizer='adam', loss='mean_squared_error')

# Print a summary to see the full architecture
Lmodel1.summary()

In [ ]:
plot_model(Lmodel1, show_shapes=True)

#### Model 2

In [ ]:
amzn_branch = create_company_model(input_amzn)
googl_branch = create_company_model(input_googl)
bll_branch = create_company_model(input_bll)
qcom_branch = create_company_model(input_qcom)

# Concatenate the outputs from all branches
# The shape will be (None, 40, 640)
concatenated_output = Concatenate()([amzn_branch, googl_branch, bll_branch, qcom_branch])

# Downsize the concatenated output with a new LSTM layer
# We use return_sequences=False to get a single output for the entire 40 timesteps.
downsized_layer = LSTM(downsized_units, return_sequences=True, name = 'lstm_conc')(concatenated_output)

# Create separate outputs for each company
amzn_final = LSTM(lstm_units, name='amzn_0')(downsized_layer)
amzn_final = Dense(1, name='amzn_output')(amzn_final)

googl_final = LSTM(lstm_units, name='googl_0')(downsized_layer)
googl_final = Dense(1, name='googl_output')(googl_final)

bll_final = LSTM(lstm_units, name='bll_0')(downsized_layer)
bll_final = Dense(1, name='bll_output')(bll_final)

qcom_final = LSTM(lstm_units, name='qcom_0')(downsized_layer)
qcom_final = Dense(1, name='qcom_output')(qcom_final)

# Define the full model
Lmodel2 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

adam = Adam(learning_rate=0.001)

# Compile the model
Lmodel2.compile(optimizer='adam', loss='mean_squared_error')

# Print a summary to see the full architecture
Lmodel2.summary()

#### Model 3

In [ ]:
amzn_branch = create_company_model(input_amzn)
googl_branch = create_company_model(input_googl)
bll_branch = create_company_model(input_bll)
qcom_branch = create_company_model(input_qcom)

# Concatenate the outputs from all branches
# The shape will be (None, 40, 640)
concatenated_output = Concatenate()([amzn_branch, googl_branch, bll_branch, qcom_branch])

# Downsize the concatenated output with a new LSTM layer
# We use return_sequences=False to get a single output for the entire 40 timesteps.
downsized_layer = LSTM(downsized_units,  return_sequences=True, name = 'lstm_conc')(concatenated_output)
downsized_layer = Dropout(dropout_rate)(downsized_layer)

# Create separate outputs for each company
amzn_final = LSTM(lstm_units, name='amzn_0')(downsized_layer)
amzn_final = Dense(1, name='amzn_output')(amzn_final)

googl_final = LSTM(lstm_units, name='googl_0')(downsized_layer)
googl_final = Dense(1, name='googl_output')(googl_final)

bll_final = LSTM(lstm_units, name='bll_0')(downsized_layer)
bll_final = Dense(1, name='bll_output')(bll_final)

qcom_final = LSTM(lstm_units, name='qcom_0')(downsized_layer)
qcom_final = Dense(1, name='qcom_output')(qcom_final)

# Define the full model
Lmodel3 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

adam = Adam(learning_rate=0.001)

# Compile the model
Lmodel3.compile(optimizer='adam', loss='mean_squared_error')

# Print a summary to see the full architecture
Lmodel3.summary()

#### Model 4

In [ ]:
amzn_branch = create_company_model(input_amzn)
googl_branch = create_company_model(input_googl)
bll_branch = create_company_model(input_bll)
qcom_branch = create_company_model(input_qcom)

# Concatenate the outputs from all branches
# The shape will be (None, 40, 640)
concatenated_output = Concatenate()([amzn_branch, googl_branch, bll_branch, qcom_branch])

# Stabilize the downsized output (None, 40, 160) to (None, 40, 160)
stabilized_layer = LSTM(stabilized_units, return_sequences=True, name = 'LSTM_conc1')(concatenated_output)
stabilized_layer = LSTM(stabilized_units, return_sequences=True, name = 'LSTM_conc2')(stabilized_layer)

# Create separate outputs for each company
amzn_final = LSTM(lstm_units, name='amzn_0')(stabilized_layer)
amzn_final = Dense(1, name='amzn_output')(amzn_final)

googl_final = LSTM(lstm_units, name='googl_0')(stabilized_layer)
googl_final = Dense(1, name='googl_output')(googl_final)

bll_final = LSTM(lstm_units, name='bll_0')(stabilized_layer)
bll_final = Dense(1, name='bll_output')(bll_final)

qcom_final = LSTM(lstm_units, name='qcom_0')(stabilized_layer)
qcom_final = Dense(1, name='qcom_output')(qcom_final)

# Define the full model
Lmodel4 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

adam = Adam(learning_rate=0.001)

# Compile the model
Lmodel4.compile(optimizer='adam', loss='mean_squared_error')

# Print a summary to see the full architecture
Lmodel4.summary()

###GRU Models

In [ ]:
# Creates independent GRU blocks for each company
def create_company_model(input_tensor):
    x = GRU(gru_units, return_sequences=True)(input_tensor)
    x = Dropout(dropout_rate)(x)
    x = GRU(gru_units, return_sequences=True)(x)
    x = Dropout(dropout_rate)(x)
    return x

####Model 1

In [ ]:
amzn_branch = create_company_model(input_amzn)
googl_branch = create_company_model(input_googl)
bll_branch = create_company_model(input_bll)
qcom_branch = create_company_model(input_qcom)

# Concatenate the outputs from all company branches
# The shape of the concatenated output will be (None, 40, 640)
concatenated_output = Concatenate()([amzn_branch, googl_branch, bll_branch, qcom_branch])


# Create separate outputs for each company as requested
amzn_final = GRU(160, name='amzn_0')(concatenated_output)
amzn_final = Dense(1, name='amzn_output')(amzn_final)

googl_final = GRU(160, name='googl_0')(concatenated_output)
googl_final = Dense(1, name='googl_output')(googl_final)

bll_final = GRU(160, name='bll_0')(concatenated_output)
bll_final = Dense(1, name='bll_output')(bll_final)

qcom_final = GRU(160, name='qcom_0')(concatenated_output)
qcom_final = Dense(1, name='qcom_output')(qcom_final)

# Define the full model with all inputs and outputs
Gmodel1 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

adam = Adam(learning_rate=0.001)

# Compile the model
Gmodel1.compile(optimizer='adam', loss='mse')

# Print a summary to see the full architecture
Gmodel1.summary()

In [ ]:
plot_model(Gmodel1, show_shapes=True)

#### Model 2

In [ ]:
amzn_branch = create_company_model(input_amzn)
googl_branch = create_company_model(input_googl)
bll_branch = create_company_model(input_bll)
qcom_branch = create_company_model(input_qcom)

# Concatenate the outputs from all branches
# The shape will be (None, 40, 640)
concatenated_output = Concatenate()([amzn_branch, googl_branch, bll_branch, qcom_branch])

# Downsize the concatenated output with a new GRU layer
# We use return_sequences=False to get a single output for the entire 40 timesteps.
downsized_layer = GRU(downsized_units, return_sequences=True, name = 'lstm_conc')(concatenated_output)

# Create separate outputs for each company
amzn_final = GRU(lstm_units, name='amzn_0')(downsized_layer)
amzn_final = Dense(1, name='amzn_output')(amzn_final)

googl_final = GRU(lstm_units, name='googl_0')(downsized_layer)
googl_final = Dense(1, name='googl_output')(googl_final)

bll_final = GRU(lstm_units, name='bll_0')(downsized_layer)
bll_final = Dense(1, name='bll_output')(bll_final)

qcom_final = GRU(lstm_units, name='qcom_0')(downsized_layer)
qcom_final = Dense(1, name='qcom_output')(qcom_final)

# Define the full model
Gmodel2 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

# Define the full model
Gmodel2 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

adam = Adam(learning_rate=0.001)

# Compile the model
Gmodel2.compile(optimizer='adam', loss='mean_squared_error')

# Print a summary to see the full architecture
Gmodel2.summary()

#### Model 3

In [ ]:
amzn_branch = create_company_model(input_amzn)
googl_branch = create_company_model(input_googl)
bll_branch = create_company_model(input_bll)
qcom_branch = create_company_model(input_qcom)

# Concatenate the outputs from all branches
# The shape will be (None, 40, 640)
concatenated_output = Concatenate()([amzn_branch, googl_branch, bll_branch, qcom_branch])

# Downsize the concatenated output with a new GRU layer
# We use return_sequences=False to get a single output for the entire 40 timesteps.
downsized_layer = GRU(downsized_units,  return_sequences=True, name = 'gru_conc')(concatenated_output)
downsized_layer = Dropout(dropout_rate)(downsized_layer)

# Create separate outputs for each company
amzn_final = GRU(gru_units, name='amzn_0')(downsized_layer)
amzn_final = Dense(1, name='amzn_output')(amzn_final)

googl_final = GRU(gru_units, name='googl_0')(downsized_layer)
googl_final = Dense(1, name='googl_output')(googl_final)

bll_final = GRU(gru_units, name='bll_0')(downsized_layer)
bll_final = Dense(1, name='bll_output')(bll_final)

qcom_final = GRU(gru_units, name='qcom_0')(downsized_layer)
qcom_final = Dense(1, name='qcom_output')(qcom_final)

# Define the full model
Gmodel3 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

adam = Adam(learning_rate=0.001)

# Compile the model
Gmodel3.compile(optimizer='adam', loss='mean_squared_error')

# Print a summary to see the full architecture
Gmodel3.summary()

#### Model 4

In [ ]:
amzn_branch = create_company_model(input_amzn)
googl_branch = create_company_model(input_googl)
bll_branch = create_company_model(input_bll)
qcom_branch = create_company_model(input_qcom)

# Concatenate the outputs from all branches
# The shape will be (None, 40, 640)
concatenated_output = Concatenate()([amzn_branch, googl_branch, bll_branch, qcom_branch])

# Stabilize the downsized output (None, 40, 160) to (None, 40, 160)
downsized_layer = GRU(stabilized_units, return_sequences=True, name = 'GRU_conc1')(concatenated_output)
stabilized_layer = GRU(stabilized_units, return_sequences=True, name = 'GRU_conc2')(downsized_layer)

# Create separate outputs for each company
amzn_final = GRU(gru_units, name='amzn_0')(stabilized_layer)
amzn_final = Dense(1, name='amzn_output')(amzn_final)

googl_final = GRU(gru_units, name='googl_0')(stabilized_layer)
googl_final = Dense(1, name='googl_output')(googl_final)

bll_final = GRU(gru_units, name='bll_0')(stabilized_layer)
bll_final = Dense(1, name='bll_output')(bll_final)

qcom_final = GRU(gru_units, name='qcom_0')(stabilized_layer)
qcom_final = Dense(1, name='qcom_output')(qcom_final)


# Define the full model
Gmodel4 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

adam = Adam(learning_rate=0.001)

# Compile the model
Gmodel4.compile(optimizer='adam', loss='mean_squared_error')

# Print a summary to see the full architecture
Gmodel4.summary()

## 4 Hybrid LSTM-GRU Neural Architecture *Models*

###LSTM -> GRU

In [ ]:
# Creates the hybrid LSTM- > GRU blocks for each company
def create_company_model(input_tensor):
    # First layer is LSTM to capture long-term dependencies
    x = LSTM(lstm_units, return_sequences=True)(input_tensor)
    x = Dropout(dropout_rate)(x)
    # Second layer is GRU for efficient processing
    x = GRU(gru_units, return_sequences=True)(x)
    x = Dropout(dropout_rate)(x)
    return x

####Model 1

In [ ]:
amzn_branch = create_company_model(input_amzn)
googl_branch = create_company_model(input_googl)
bll_branch = create_company_model(input_bll)
qcom_branch = create_company_model(input_qcom)

# Concatenate the outputs from all company branches
# The shape of the concatenated output will be (None, 40, 640)
concatenated_output = Concatenate()([amzn_branch, googl_branch, bll_branch, qcom_branch])


# Create separate outputs for each company as requested
amzn_final = GRU(160, name='amzn_0')(concatenated_output)
amzn_final = Dense(1, name='amzn_output')(amzn_final)

googl_final = GRU(160, name='googl_0')(concatenated_output)
googl_final = Dense(1, name='googl_output')(googl_final)

bll_final = GRU(160, name='bll_0')(concatenated_output)
bll_final = Dense(1, name='bll_output')(bll_final)

qcom_final = GRU(160, name='qcom_0')(concatenated_output)
qcom_final = Dense(1, name='qcom_output')(qcom_final)

# Define the full model with all inputs and outputs
LGmodel1 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

# Compile the model
LGmodel1.compile(optimizer='adam', loss='mean_squared_error')

# Print a summary to see the full architecture
LGmodel1.summary()

#### Model 2

In [ ]:
amzn_branch = create_company_model(input_amzn)
googl_branch = create_company_model(input_googl)
bll_branch = create_company_model(input_bll)
qcom_branch = create_company_model(input_qcom)

# Concatenate the outputs from all branches
# The shape will be (None, 40, 640)
concatenated_output = Concatenate()([amzn_branch, googl_branch, bll_branch, qcom_branch])

# Downsize the concatenated output with a new GRU layer
# We use return_sequences=False to get a single output for the entire 40 timesteps.
downsized_layer = GRU(downsized_units, return_sequences=True, name = 'GRU_conc')(concatenated_output)


# Create separate outputs for each company
amzn_final = GRU(lstm_units, name='amzn_0')(downsized_layer)
amzn_final = Dense(1, name='amzn_output')(amzn_final)

googl_final = GRU(lstm_units, name='googl_0')(downsized_layer)
googl_final = Dense(1, name='googl_output')(googl_final)

bll_final = GRU(lstm_units, name='bll_0')(downsized_layer)
bll_final = Dense(1, name='bll_output')(bll_final)

qcom_final = GRU(lstm_units, name='qcom_0')(downsized_layer)
qcom_final = Dense(1, name='qcom_output')(qcom_final)

# Define the full model
LGmodel2 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

# Compile the model
LGmodel2.compile(optimizer='adam', loss='mean_squared_error')

# Print a summary to see the full architecture
LGmodel2.summary()

#### Model 3

In [ ]:
amzn_branch = create_company_model(input_amzn)
googl_branch = create_company_model(input_googl)
bll_branch = create_company_model(input_bll)
qcom_branch = create_company_model(input_qcom)

# Concatenate the outputs from all branches
# The shape will be (None, 40, 640)
concatenated_output = Concatenate()([amzn_branch, googl_branch, bll_branch, qcom_branch])

# Downsize the concatenated output with a new GRU layer
# We use return_sequences=False to get a single output for the entire 40 timesteps.
downsized_layer = GRU(downsized_units,  return_sequences=True, name = 'gru_conc')(concatenated_output)
downsized_layer = Dropout(dropout_rate)(downsized_layer)

# Create separate outputs for each company
amzn_final = GRU(gru_units, name='amzn_0')(downsized_layer)
amzn_final = Dense(1, name='amzn_output')(amzn_final)

googl_final = GRU(gru_units, name='googl_0')(downsized_layer)
googl_final = Dense(1, name='googl_output')(googl_final)

bll_final = GRU(gru_units, name='bll_0')(downsized_layer)
bll_final = Dense(1, name='bll_output')(bll_final)

qcom_final = GRU(gru_units, name='qcom_0')(downsized_layer)
qcom_final = Dense(1, name='qcom_output')(qcom_final)

# Define the full model
LGmodel3 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

# Compile the model
LGmodel3.compile(optimizer='adam', loss='mean_squared_error')

# Print a summary to see the full architecture
LGmodel3.summary()

#### Model 4

In [ ]:
amzn_branch = create_company_model(input_amzn)
googl_branch = create_company_model(input_googl)
bll_branch = create_company_model(input_bll)
qcom_branch = create_company_model(input_qcom)

# Concatenate the outputs from all branches
# The shape will be (None, 40, 640)
concatenated_output = Concatenate()([amzn_branch, googl_branch, bll_branch, qcom_branch])

# Stabilize the downsized output (None, 40, 160) to (None, 40, 160)
downsized_layer = GRU(stabilized_units, return_sequences=True, name = 'GRU_conc1')(concatenated_output)
stabilized_layer = GRU(stabilized_units, return_sequences=True, name = 'GRU_conc2')(downsized_layer)

# Create separate outputs for each company
amzn_final = GRU(gru_units, name='amzn_0')(stabilized_layer)
amzn_final = Dense(1, name='amzn_output')(amzn_final)

googl_final = GRU(gru_units, name='googl_0')(stabilized_layer)
googl_final = Dense(1, name='googl_output')(googl_final)

bll_final = GRU(gru_units, name='bll_0')(stabilized_layer)
bll_final = Dense(1, name='bll_output')(bll_final)

qcom_final = GRU(gru_units, name='qcom_0')(stabilized_layer)
qcom_final = Dense(1, name='qcom_output')(qcom_final)

# Define the full model
LGmodel4 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

# Compile the model
LGmodel4.compile(optimizer='adam', loss='mean_squared_error')

# Print a summary to see the full architecture
LGmodel4.summary()

###GRU -> LSTM

In [ ]:
# Creates the hybrid GRU -> LSTM blocks for each company
def create_company_model(input_tensor):
    # First layer is GRU for efficient initial processing
    x = GRU(gru_units, return_sequences=True)(input_tensor)
    x = Dropout(dropout_rate)(x)
    # Second layer is LSTM to refine learned dependencies
    x = LSTM(lstm_units, return_sequences=True)(x)
    x = Dropout(dropout_rate)(x)
    return x


####Model 1

In [ ]:
amzn_branch = create_company_model(input_amzn)
googl_branch = create_company_model(input_googl)
bll_branch = create_company_model(input_bll)
qcom_branch = create_company_model(input_qcom)

# Concatenate the outputs from all company branches
# The shape of the concatenated output will be (None, 40, 640)
concatenated_output = Concatenate()([amzn_branch, googl_branch, bll_branch, qcom_branch])

# Create separate outputs for each company as requested
amzn_final = LSTM(lstm_units, name='amzn_0')(concatenated_output)
amzn_final = Dense(1, name='amzn_output')(amzn_final)

googl_final = LSTM(lstm_units, name='googl_0')(concatenated_output)
googl_final = Dense(1, name='googl_output')(googl_final)

bll_final = LSTM(lstm_units, name='bll_0')(concatenated_output)
bll_final = Dense(1, name='bll_output')(bll_final)

qcom_final = LSTM(lstm_units, name='qcom_0')(concatenated_output)
qcom_final = Dense(1, name='qcom_output')(qcom_final)


# Define the full model with all inputs and outputs
GLmodel1 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

# Compile the model
GLmodel1.compile(optimizer='adam', loss='mean_squared_error')

# Print a summary to see the full architecture
GLmodel1.summary()

#### Model 2

In [ ]:
amzn_branch = create_company_model(input_amzn)
googl_branch = create_company_model(input_googl)
bll_branch = create_company_model(input_bll)
qcom_branch = create_company_model(input_qcom)

# Concatenate the outputs from all branches
# The shape will be (None, 40, 640)
concatenated_output = Concatenate()([amzn_branch, googl_branch, bll_branch, qcom_branch])

# Downsize the concatenated output with a new LSTM layer
# We use return_sequences=False to get a single output for the entire 40 timesteps.
downsized_layer = LSTM(downsized_units, return_sequences=True, name = 'lstm_conc')(concatenated_output)

# Create separate outputs for each company
amzn_final = LSTM(lstm_units, name='amzn_0')(downsized_layer)
amzn_final = Dense(1, name='amzn_output')(amzn_final)

googl_final = LSTM(lstm_units, name='googl_0')(downsized_layer)
googl_final = Dense(1, name='googl_output')(googl_final)

bll_final = LSTM(lstm_units, name='bll_0')(downsized_layer)
bll_final = Dense(1, name='bll_output')(bll_final)

qcom_final = LSTM(lstm_units, name='qcom_0')(downsized_layer)
qcom_final = Dense(1, name='qcom_output')(qcom_final)

# Define the full model
GLmodel2 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

# Compile the model
GLmodel2.compile(optimizer='adam', loss='mean_squared_error')

# Print a summary to see the full architecture
GLmodel2.summary()

#### Model 3

In [ ]:
amzn_branch = create_company_model(input_amzn)
googl_branch = create_company_model(input_googl)
bll_branch = create_company_model(input_bll)
qcom_branch = create_company_model(input_qcom)

# Concatenate the outputs from all branches
# The shape will be (None, 40, 640)
concatenated_output = Concatenate()([amzn_branch, googl_branch, bll_branch, qcom_branch])

# Downsize the concatenated output with a new LSTM layer
# We use return_sequences=False to get a single output for the entire 40 timesteps.
downsized_layer = LSTM(downsized_units,  return_sequences=True, name = 'lstm_conc')(concatenated_output)
downsized_layer = Dropout(dropout_rate)(downsized_layer)

# Create separate outputs for each company
amzn_final = LSTM(lstm_units, name='amzn_0')(downsized_layer)
amzn_final = Dense(1, name='amzn_output')(amzn_final)

googl_final = LSTM(lstm_units, name='googl_0')(downsized_layer)
googl_final = Dense(1, name='googl_output')(googl_final)

bll_final = LSTM(lstm_units, name='bll_0')(downsized_layer)
bll_final = Dense(1, name='bll_output')(bll_final)

qcom_final = LSTM(lstm_units, name='qcom_0')(downsized_layer)
qcom_final = Dense(1, name='qcom_output')(qcom_final)

# Define the full model
GLmodel3 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

# Compile the model
GLmodel3.compile(optimizer='adam', loss='mean_squared_error')

# Print a summary to see the full architecture
GLmodel3.summary()

#### Model 4

In [ ]:
amzn_branch = create_company_model(input_amzn)
googl_branch = create_company_model(input_googl)
bll_branch = create_company_model(input_bll)
qcom_branch = create_company_model(input_qcom)

# Concatenate the outputs from all branches
# The shape will be (None, 40, 640)
concatenated_output = Concatenate()([amzn_branch, googl_branch, bll_branch, qcom_branch])

# Stabilize the downsized output (None, 40, 160) to (None, 40, 160)
stabilized_layer = LSTM(stabilized_units, return_sequences=True, name = 'LSTM_conc1')(concatenated_output)
stabilized_layer = LSTM(stabilized_units, return_sequences=True, name = 'LSTM_conc2')(stabilized_layer)

# Create separate outputs for each company
amzn_final = LSTM(lstm_units, name='amzn_0')(stabilized_layer)
amzn_final = Dense(1, name='amzn_output')(amzn_final)

googl_final = LSTM(lstm_units, name='googl_0')(stabilized_layer)
googl_final = Dense(1, name='googl_output')(googl_final)

bll_final = LSTM(lstm_units, name='bll_0')(stabilized_layer)
bll_final = Dense(1, name='bll_output')(bll_final)

qcom_final = LSTM(lstm_units, name='qcom_0')(stabilized_layer)
qcom_final = Dense(1, name='qcom_output')(qcom_final)

# Define the full model
GLmodel4 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

# Compile the model
GLmodel4.compile(optimizer='adam', loss='mean_squared_error')

# Print a summary to see the full architecture
GLmodel4.summary()

###LSTM + GRU in parallel

In [ ]:
# Creates the hybrid parallel LSTM + GRU Architecture
def create_company_model(input_tensor):
    # GRU branch for efficient initial processing
    gru_branch = GRU(gru_units, activation='relu', return_sequences=True)(input_tensor)
    gru_branch = Dropout(dropout_rate)(gru_branch)

    # LSTM branch to capture long-term dependencies in parallel
    lstm_branch = LSTM(lstm_units, activation='relu', return_sequences=True)(input_tensor)
    lstm_branch = Dropout(dropout_rate)(lstm_branch)

    # Concatenate the outputs from the two parallel branches
    x = Concatenate()([gru_branch, lstm_branch])

    return x

####Model 1

In [ ]:
amzn_branch = create_company_model(input_amzn)
googl_branch = create_company_model(input_googl)
bll_branch = create_company_model(input_bll)
qcom_branch = create_company_model(input_qcom)

# Concatenate the outputs from all company branches
# The shape of the concatenated output will be (None, 40, 640)
concatenated_output = Concatenate()([amzn_branch, googl_branch, bll_branch, qcom_branch])


# Create separate outputs for each company as requested
amzn_final = GRU(160, name='amzn_0')(concatenated_output)
amzn_final = Dense(1, name='amzn_output')(amzn_final)

googl_final = GRU(160, name='googl_0')(concatenated_output)
googl_final = Dense(1, name='googl_output')(googl_final)

bll_final = GRU(160, name='bll_0')(concatenated_output)
bll_final = Dense(1, name='bll_output')(bll_final)

qcom_final = GRU(160, name='qcom_0')(concatenated_output)
qcom_final = Dense(1, name='qcom_output')(qcom_final)

# Define the full model with all inputs and outputs
LGmodel1 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

# Define the full model with all inputs and outputs
PARmodel1 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

# Compile the model
PARmodel1.compile(optimizer='adam', loss='mean_squared_error')

# Print a summary to see the full architecture
PARmodel1.summary()

####Model 2

In [ ]:
amzn_branch = create_company_model(input_amzn)
googl_branch = create_company_model(input_googl)
bll_branch = create_company_model(input_bll)
qcom_branch = create_company_model(input_qcom)

# Concatenate the outputs from all branches
# The shape will be (None, 40, 640)
concatenated_output = Concatenate()([amzn_branch, googl_branch, bll_branch, qcom_branch])

# Downsize the concatenated output with a new GRU layer
# We use return_sequences=False to get a single output for the entire 40 timesteps.
downsized_layer = GRU(downsized_units, return_sequences=True, name = 'lstm_conc')(concatenated_output)


# Create separate outputs for each company
amzn_final = GRU(lstm_units, name='amzn_0')(downsized_layer)
amzn_final = Dense(1, name='amzn_output')(amzn_final)

googl_final = GRU(lstm_units, name='googl_0')(downsized_layer)
googl_final = Dense(1, name='googl_output')(googl_final)

bll_final = GRU(lstm_units, name='bll_0')(downsized_layer)
bll_final = Dense(1, name='bll_output')(bll_final)

qcom_final = GRU(lstm_units, name='qcom_0')(downsized_layer)
qcom_final = Dense(1, name='qcom_output')(qcom_final)

# Define the full model
PARmodel2 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

# Compile the model
PARmodel2.compile(optimizer='adam', loss='mean_squared_error')

# Print a summary to see the full architecture
PARmodel2.summary()

####Model 3

In [ ]:
amzn_branch = create_company_model(input_amzn)
googl_branch = create_company_model(input_googl)
bll_branch = create_company_model(input_bll)
qcom_branch = create_company_model(input_qcom)

# Concatenate the outputs from all branches
# The shape will be (None, 40, 640)
concatenated_output = Concatenate()([amzn_branch, googl_branch, bll_branch, qcom_branch])

# Downsize the concatenated output with a new GRU layer
# We use return_sequences=False to get a single output for the entire 40 timesteps.
downsized_layer = GRU(downsized_units,  return_sequences=True, name = 'gru_conc')(concatenated_output)
downsized_layer = Dropout(dropout_rate)(downsized_layer)

# Create separate outputs for each company
amzn_final = GRU(gru_units, name='amzn_0')(downsized_layer)
amzn_final = Dense(1, name='amzn_output')(amzn_final)

googl_final = GRU(gru_units, name='googl_0')(downsized_layer)
googl_final = Dense(1, name='googl_output')(googl_final)

bll_final = GRU(gru_units, name='bll_0')(downsized_layer)
bll_final = Dense(1, name='bll_output')(bll_final)

qcom_final = GRU(gru_units, name='qcom_0')(downsized_layer)
qcom_final = Dense(1, name='qcom_output')(qcom_final)

# Define the full model
PARmodel3 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

# Compile the model
PARmodel3.compile(optimizer='adam', loss='mean_squared_error')

# Print a summary to see the full architecture
PARmodel3.summary()

####Model 4

In [ ]:
amzn_branch = create_company_model(input_amzn)
googl_branch = create_company_model(input_googl)
bll_branch = create_company_model(input_bll)
qcom_branch = create_company_model(input_qcom)

# Concatenate the outputs from all branches
# The shape will be (None, 40, 640)
concatenated_output = Concatenate()([amzn_branch, googl_branch, bll_branch, qcom_branch])

# Stabilize the downsized output (None, 40, 160) to (None, 40, 160)
downsized_layer = GRU(stabilized_units, return_sequences=True, name = 'GRU_conc1')(concatenated_output)
stabilized_layer = GRU(stabilized_units, return_sequences=True, name = 'GRU_conc2')(downsized_layer)

# Create separate outputs for each company
amzn_final = GRU(gru_units, name='amzn_0')(stabilized_layer)
amzn_final = Dense(1, name='amzn_output')(amzn_final)

googl_final = GRU(gru_units, name='googl_0')(stabilized_layer)
googl_final = Dense(1, name='googl_output')(googl_final)

bll_final = GRU(gru_units, name='bll_0')(stabilized_layer)
bll_final = Dense(1, name='bll_output')(bll_final)

qcom_final = GRU(gru_units, name='qcom_0')(stabilized_layer)
qcom_final = Dense(1, name='qcom_output')(qcom_final)

# Define the full model
PARmodel4 = Model(
    inputs=[input_amzn, input_googl, input_bll, input_qcom],
    outputs=[amzn_final, googl_final, bll_final, qcom_final]
)

# Compile the model
PARmodel4.compile(optimizer='adam', loss='mean_squared_error')

# Print a summary to see the full architecture
PARmodel4.summary()

##Min Max Normalisation

In [ ]:
# Visualisation purposes
Dclose = data['Close']
eps = 1e-8

# Store the original Min/Max values
Dclose_min = Dclose.min()
Dclose_max = Dclose.max()

# Min/Max range
normalized_min = 0.0
normalized_max = 1.0

# Min-max normalization
normalized_vis = (Dclose - Dclose.min()) / (Dclose.max() - Dclose.min() + eps)
print("Normalized:", normalized_vis.head())

# Store Scalar min/Max for future data
scaler_min, scaler_max = data.min(), data.max()

In [ ]:
# Plot the closing prices
plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(10, 6))
for ticker in tickers:
    plt.plot(normalized_vis[ticker], label=ticker, color=colors.get(ticker, None))

plt.title('Closing Prices for Selected Stocks (2010-2022)', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Closing Price (USD)', fontsize=12)
plt.legend(title='Ticker', fontsize=10)
plt.grid(True)
plt.tight_layout()
plt.savefig('close_prices_plot.png')
plt.show()

Visualization of the normalize close price time-series data of the selected four companies

##Date Segmentation (Timestep Grouping)

In [ ]:
Dclose = data['Close'].copy()
Dclose.index = pd.to_datetime(Dclose.index)
Dclose = Dclose.sort_index()

# Define the cutoff date for train/test
cutoff_date = pd.Timestamp('2019-09-17')  # example: first 9 years 9 months for training or 80% of the data

In this case, the data is segmented as it is time series. a random split of all data wouldnt work as recent data shouldnt be used to make the predictions. Insead the data is split at the date 17/09/2019 across the 12 years to get an 80/20 split. 80% being older, historical data and the 20% being more recent.

In [ ]:

# Split
train_df = Dclose[Dclose.index <= cutoff_date].copy()
test_df  = Dclose[Dclose.index >  cutoff_date].copy()

tmins = train_df.min()  # pd.Series indexed by ticker
tmaxs = train_df.max()

## Normalization
# Scale train/test using train stats
train_norm = (train_df - tmins) / (tmaxs - tmins + eps)
test_norm  = (test_df  - tmins) / (tmaxs - tmins + eps)

# Convert to numpy for time-step segmentation
train_prices = train_norm.to_numpy()
test_prices = test_norm.to_numpy()

# inverse-scale function for arrays/series later
def inverse_scale(y, tmin, tmax):
    # y can be np.ndarray or pd.Series of normalized values
    return y * (tmax - tmin) + tmin


print(f"\nCreating time-step sequences with a window of {timesteps}...")
# Create training sequences (X) and corresponding labels (y)
X_train, y_train = [], []
for ticker in tickers:
    X_train_ticker, y_train_ticker = [], []
    train_prices_ticker = train_norm[ticker].to_numpy()
    for i in range(len(train_prices_ticker) - timesteps):
        X_train_ticker.append(train_prices_ticker[i:i+timesteps])
        y_train_ticker.append(train_prices_ticker[i+timesteps])

    # Reshape for the models input layer and add to the list
    X_train.append(np.array(X_train_ticker).reshape(-1, timesteps, 1))

    # y_train will have shape (samples, 1) for each company
    y_train.append(np.array(y_train_ticker).reshape(-1, 1))

# Create testing sequences (X) and corresponding labels (y)
X_test, y_test = [], []
for ticker in tickers:
    X_test_ticker, y_test_ticker = [], []
    train_prices_ticker = train_norm[ticker].to_numpy()
    test_prices_ticker = test_norm[ticker].to_numpy()

    # For test, use the last 'timesteps' from training as initial input
    test_series = np.concatenate([train_prices_ticker[-timesteps:], test_prices_ticker])
    for i in range(len(test_prices_ticker)):
        X_test_ticker.append(test_series[i:i+timesteps])
        y_test_ticker.append(test_prices_ticker[i])

    # Reshape for the GRU input layer and add to the list
    X_test.append(np.array(X_test_ticker).reshape(-1, timesteps, 1))

    # y_test will have shape (samples, 1) for each company
    y_test.append(np.array(y_test_ticker).reshape(-1, 1))

print(f"Training samples: {X_train[0].shape[0]}")
print(f"Testing samples: {X_test[0].shape[0]}")

In [ ]:
X_train = np.asarray(X_train)
X_test = np.asarray(X_test)
y_train = np.asarray(y_train)
y_test = np.asarray(y_test)

In [ ]:
display("X_train shape:", X_train.shape, "X_test shape:", X_test.shape, "y_train shape:", y_train.shape, "y_test shape:", y_test.shape)

In [ ]:
display("X_train shape:", X_train.shape, "X_test shape:", X_test.shape, "y_train shape:", y_train.shape, "y_test shape:", y_test.shape)

In [ ]:
# Unpack per ticker
X_train_amzn, X_train_googl, X_train_ball, X_train_qcom = X_train
X_test_amzn,  X_test_googl,  X_test_ball,  X_test_qcom  = X_test
y_train_amzn, y_train_googl, y_train_ball, y_train_qcom = y_train
y_test_amzn,  y_test_googl,  y_test_ball,  y_test_qcom  = y_test

# Sanity checks
print("X_train_amzn:", X_train_amzn.shape, "y_train_amzn:", y_train_amzn.shape)
print("X_train_googl:", X_train_googl.shape, "y_train_googl:", y_train_googl.shape)
print("X_train_ball:", X_train_ball.shape, "y_train_ball:", y_train_ball.shape)
print("X_train_qcom:", X_train_qcom.shape, "y_train_qcom:", y_train_qcom.shape)

print("X_test_amzn:", X_test_amzn.shape, "y_test_amzn:", y_test_amzn.shape)
print("X_test_googl:", X_test_googl.shape, "y_test_googl:", y_test_googl.shape)
print("X_test_ball:", X_test_ball.shape, "y_test_ball:", y_test_ball.shape)
print("X_test_qcom:", X_test_qcom.shape, "y_test_qcom:", y_test_qcom.shape)

# Optional: assert expected shapes
assert X_train_amzn.shape == (2403, 40, 1)
assert X_test_amzn.shape  == (601, 40, 1)
assert y_train_amzn.shape == (2403, 1)
assert y_test_amzn.shape  == (601, 1)

In [ ]:
data = {
    "AMZN": {
        "X_train": X_train_amzn, "y_train": y_train_amzn,
        "X_test":  X_test_amzn,  "y_test":  y_test_amzn,
    },
    "GOOGL": {
        "X_train": X_train_googl, "y_train": y_train_googl,
        "X_test":  X_test_googl,  "y_test":  y_test_googl,
    },
    "BALL": {
        "X_train": X_train_ball, "y_train": y_train_ball,
        "X_test":  X_test_ball,  "y_test":  y_test_ball,
    },
    "QCOM": {
        "X_train": X_train_qcom, "y_train": y_train_qcom,
        "X_test":  X_test_qcom,  "y_test":  y_test_qcom,
    },
}

In [ ]:
num_trading_days = len(close_prices['AMZN'].loc['2019-09-18':'2022-02-03'])
print(num_trading_days)

In [ ]:
for i, ticker in enumerate(tickers):
    print(f"\nTicker: {ticker}")
    print("Number of input sequences (X):", len(X_train[i]))
    print("Number of targets (y):", len(y_train[i]))
    print("Do they match? ->", len(X_train[i]) == len(y_train[i]))

In [ ]:
# Merge all tickers' sequences
X_all = np.concatenate(X_train, axis=0)
y_all = np.concatenate(y_train, axis=0)

print("Number of input sequences (X):", len(X_all))
print("Number of targets (y):", len(y_all))
print("Do they match? ->", len(X_all) == len(y_all))

## Establishing Evaluation Metrics

In [ ]:

# MAPE
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# RMSPE
def rmspe(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.sqrt(np.mean(((y_true - y_pred) / y_true) ** 2)) * 100

#RMDPE
def rmdpe(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.sqrt(np.median(((y_true - y_pred) / y_true) ** 2)) * 100

## Prior research Training

In [ ]:

# Create a dictionary to hold all models
models_to_train = {
    "Lmodel1": Lmodel1,
    # "Lmodel2": Lmodel2,
    # "Lmodel3": Lmodel3,
    # "Lmodel4": Lmodel4,
    # "Gmodel1": Gmodel1,
    # "Gmodel2": Gmodel2,
    # "Gmodel3": Gmodel3,
    # "Gmodel4": Gmodel4
}

In [ ]:
X_train[0].shape


In [ ]:
# Create a dictionary to store the training history for each model
training_histories = {}

# Build ordered inputs/targets from your nested dict
tickers = ["AMZN", "GOOGL", "BALL", "QCOM"]
X_inputs = [data[t]["X_train"] for t in tickers]
y_targets = [data[t]["y_train"] for t in tickers]

for model_name, model in models_to_train.items():
    print(f"\n--- Training {model_name} ---")
    history = model.fit(
        x=X_inputs,
        y=y_targets,
        epochs=30,
        batch_size=32,
        validation_split=0.2,
        verbose=1,
        # callbacks=[...]  # optional
    )
    training_histories[model_name] = history

    # Manual evaluation on the hold-out part of the training set (same split ratio)
    val_size = int(X_inputs[0].shape[0] * 0.2)
    if val_size > 0:
        X_val_split = [xi[-val_size:] for xi in X_inputs]
        y_val_split = [yi[-val_size:] for yi in y_targets]

        eval_vals = model.evaluate(X_val_split, y_val_split, verbose=0)

        # Handle single vs multi-output models
        if isinstance(eval_vals, (list, tuple)):
            # Keras evaluate returns: [total_loss, out1_loss, out2_loss, ..., metrics...]
            names = ["loss"] + model.metrics_names[1:len(eval_vals)]
            eval_report = {k: float(v) for k, v in zip(names, eval_vals)}
        else:
            eval_report = {"loss": float(eval_vals)}

        print("Eval on val split:", {k: round(v, 6) for k, v in eval_report.items()})

    print(f"Training for {model_name} complete.")

In [ ]:

for t in ["AMZN", "GOOGL", "BALL", "QCOM"]:
    y = data[t]["y_train"]
    print(t, "y NaN:", np.isnan(y).any(), "Inf:", np.isinf(y).any())

# Current research Training

In [ ]:
# Create a dictionary to hold all models
current_models_to_train = {
    "LGmodel1": LGmodel1, #LSTM -> GRU
    "LGmodel2": LGmodel2,
    "LGodel3": LGmodel3,
    "LGmdel4": LGmodel4,
    "GLmoel1": GLmodel1, #GRU -> LSTM
    "GLmodl2": GLmodel2,
    "GLmode3": GLmodel3,
    "GLmodel": GLmodel4,
    "PARmodel1": PARmodel1,   #Parallel LSTM + GRU
    "PARmodel2": PARmodel2,
    "PARmodel3": PARmodel3,
    "PARmodel4": PARmodel4
}

In [ ]:

# Create a dictionary to store the training history for each model
current_training_histories = {}

# Ensure ordered model inputs [amzn, googl, bll, qcom]
# Ensure ordered outputs  [amzn_output, googl_output, bll_output, qcom_output]
X_inputs = [X_train[0], X_train[1], X_train[2], X_train[3]]
y_targets = [y_train[0], y_train[1], y_train[2], y_train[3]]
X_test     = [X_test[0],  X_test[1],  X_test[2],  X_test[3]]
y_test     = [y_test[0],  y_test[1],  y_test[2],  y_test[3]]


# Train each model and store its history
for model_name, model in current_models_to_train.items():
    print(f"\n--- Training {model_name} ---")
    history = model.fit(
        x=X_inputs,
        y=y_targets,
        epochs=10,
        batch_size=32,
        validation_data=(X_test, y_test),
        verbose=1
    )
    training_histories[model_name] = history

    print(f"Training for {model_name} complete.")

##Plottting previous research

In [ ]:

# Group models into LSTM and GRU
lstm_models = {k: v for k, v in models_to_train.items() if k.startswith("Lmodel")}
gru_models  = {k: v for k, v in models_to_train.items() if k.startswith("Gmodel")}

# After you create X_test and y_test lists in AMZN, GOOGL, BALL, QCOM order
ticker_order = ["AMZN", "GOOGL", "BALL", "QCOM"]

X_test_dict = {t: X_test[i] for i, t in enumerate(ticker_order)}
y_test_dict = {t: y_test[i] for i, t in enumerate(ticker_order)}

# Helper function for denormalized predictions
def get_predictions(model, X_dict, y_dict):
    y_preds = model.predict(
        [X_dict["AMZN"], X_dict["GOOGL"], X_dict["BALL"], X_dict["QCOM"]],
        verbose=0,
    )
    # y_preds is a list [amzn_pred, googl_pred, ball_pred, qcom_pred]
    # Ensure each is (n_samples,)
    preds = {}
    for ticker, yp in zip(["AMZN", "GOOGL", "BALL", "QCOM"], y_preds):
        yp = np.asarray(yp)
        # If model accidentally outputs sequences (n, T, 1), take last step
        if yp.ndim == 3:
            yp = yp[:, -1, 0]
        elif yp.ndim == 2:
            yp = yp[:, 0]
        preds[ticker] = yp

    results = {}
    for ticker in ["AMZN", "GOOGL", "BALL", "QCOM"]:
        y_true = np.asarray(y_dict[ticker]).squeeze()
        # Align shapes to (n_samples,)
        if y_true.ndim == 2:
            y_true = y_true[:, 0]
        # inverse scale
        tmin, tmax = tmins[ticker], tmaxs[ticker]
        y_true_denorm = inverse_scale(y_true, tmin, tmax)
        y_pred_denorm = inverse_scale(preds[ticker], tmin, tmax)
        results[ticker] = (y_true_denorm, y_pred_denorm)  # tuple of 1D arrays
    return results

# --- 1. LSTMs for AMZN + GOOGL ---
plt.figure(figsize=(12,6))
for model_name, model in lstm_models.items():
    preds = get_predictions(model, X_test_dict, y_test_dict)
    plt.plot(preds["AMZN"][1], alpha=0.7, label=f"{model_name} - AMZN")
    plt.plot(preds["GOOGL"][1], alpha=0.7, label=f"{model_name} - GOOGL")
# Plot actuals on top
plt.plot(preds["AMZN"][0], "k--", linewidth=2.2, label="AMZN Actual")
plt.plot(preds["GOOGL"][0], "r--", linewidth=2.2, label="GOOGL Actual")
plt.title("All LSTM Models - AMZN & GOOGL")
plt.xlabel("Time Steps")
plt.ylabel("Stock Price")
plt.legend(ncol=2, loc="best", frameon=True)
plt.show()

# --- 2. GRUs for AMZN + GOOGL ---
plt.figure(figsize=(12,6))
for model_name, model in gru_models.items():
    preds = get_predictions(model, X_test_dict, y_test_dict)
    plt.plot(preds["AMZN"][1], alpha=0.7, label=f"{model_name} - AMZN")
    plt.plot(preds["GOOGL"][1], alpha=0.7, label=f"{model_name} - GOOGL")
plt.plot(preds["AMZN"][0], "k--", linewidth=2.2, label="AMZN Actual")
plt.plot(preds["GOOGL"][0], "r--", linewidth=2.2, label="GOOGL Actual")
plt.title("All GRU Models - AMZN & GOOGL")
plt.xlabel("Time Steps")
plt.ylabel("Stock Price")
plt.legend(ncol=2, loc="best", frameon=True)
plt.show()

# --- 3. LSTMs for BALL + QCOM ---
plt.figure(figsize=(12,6))
for model_name, model in lstm_models.items():
    preds = get_predictions(model, X_test_dict, y_test_dict)
    plt.plot(preds["BALL"][1], alpha=0.7, label=f"{model_name} - BALL")
    plt.plot(preds["QCOM"][1], alpha=0.7, label=f"{model_name} - QCOM")
plt.plot(preds["BALL"][0], "k--", linewidth=2.2, label="BALL Actual")
plt.plot(preds["QCOM"][0], "r--", linewidth=2.2, label="QCOM Actual")
plt.title("All LSTM Models - BALL & QCOM")
plt.xlabel("Time Steps")
plt.ylabel("Stock Price")
plt.legend(ncol=2, loc="best", frameon=True)
plt.show()

# --- 4. GRUs for BALL + QCOM ---
plt.figure(figsize=(12,6))
for model_name, model in gru_models.items():
    preds = get_predictions(model, X_test_dict, y_test_dict)
    plt.plot(preds["BALL"][1], alpha=0.7, label=f"{model_name} - BALL")
    plt.plot(preds["QCOM"][1], alpha=0.7, label=f"{model_name} - QCOM")
plt.plot(preds["BALL"][0], "k--", linewidth=2.2, label="BALL Actual")
plt.plot(preds["QCOM"][0], "r--", linewidth=2.2, label="QCOM Actual")
plt.title("All GRU Models - BALL & QCOM")
plt.xlabel("Time Steps")
plt.ylabel("Stock Price")
plt.legend(ncol=2, loc="best", frameon=True)
plt.show()

In [ ]:
def last_value_baseline(X):  # X: (n, 40, 1)
    return X[:, -1, 0]

for i, t in enumerate(["AMZN","GOOGL","BALL","QCOM"]):
    yb = last_value_baseline(X_test[i])
    yt = y_test[i][:, 0]
    print(t, "MAE(norm):", float(np.mean(np.abs(yt - yb))))

In [ ]:

def model_mae_norm(model, X_test):
    # Predict in normalized space
    preds = model.predict([X_test[0], X_test[1], X_test[2], X_test[3]], verbose=0)
    maes = {}
    for i, t in enumerate(["AMZN","GOOGL","BALL","QCOM"]):
        y_pred = np.asarray(preds[i]).squeeze()
        y_true = y_test[i].squeeze()
        if y_pred.ndim == 2:  # (n,1) -> (n,)
            y_pred = y_pred[:, 0]
        if y_true.ndim == 2:
            y_true = y_true[:, 0]
        maes[t] = float(np.mean(np.abs(y_true - y_pred)))
    return maes

for name, model in models_to_train.items():
    maes = model_mae_norm(model, X_test)
    print(name, "MAE(norm):", {k: round(v, 5) for k, v in maes.items()})

In [ ]:
preds = get_predictions(list(models_to_train.values())[0], X_test_dict, y_test_dict)
t="AMZN"
yt, yp = preds[t]
print("AMZN head true/pred:", list(zip(yt[:5], yp[:5])))

#Plotting Current Research

In [ ]:
# Group models into LSTM and GRU
model_groups = {
    "LSTM → GRU": {k: v for k, v in models_to_train.items() if k.startswith("LGmodel")},
    "GRU → LSTM": {k: v for k, v in models_to_train.items() if k.startswith("GLmodel")},
    "Parallel (LSTM + GRU)": {k: v for k, v in models_to_train.items() if k.startswith("PARmodel")},
}

#Model Evaluation

#Past research

In [ ]:

# Evaluate helper
def eval_model(model, X_test_dict, y_test_dict, tickers=("AMZN","GOOGL","BALL","QCOM")):
    preds = get_predictions(model, X_test_dict, y_test_dict)
    rows = []
    for t in tickers:
        y_true, y_pred = preds[t]
        rows.append({
            "ticker": t,
            "MAPE":  mape(y_true, y_pred),
            "RMSPE": rmspe(y_true, y_pred),
            "RMDPE": rmdpe(y_true, y_pred),
        })
    return pd.DataFrame(rows)

# Run across your model groups (adjust dicts to your setup)
all_rows = []
for model_name, model in models_to_train.items():
    df = eval_model(model, X_test_dict, y_test_dict)
    df.insert(0, "model", model_name)
    all_rows.append(df)

metrics_df = pd.concat(all_rows, ignore_index=True)

# Pretty print per model
for name in metrics_df["model"].unique():
    print(f"\n== {name} ==")
    print(metrics_df[metrics_df["model"] == name][["ticker","MAPE","RMSPE","RMDPE"]].round(3).to_string(index=False))

# Save to CSV for later analysis
metrics_df.to_csv("metrics_denorm.csv", index=False)